# Taller 1 — Consumo Automatizado de APIs

**Asignatura:** MLY1101 — Machine Learning  
**Nombre del estudiante:** Christian Tapia  
**Sección:** MLY1101_001V  
**Fecha:** 13-08-2026

## Pregunta u objetivo

> **¿Qué información puede recopilarse sobre videojuegos free-to-play, sus plataformas y el contenido jugable que ofrecen?**

La pregunta se aborda en tres niveles complementarios. **No se responde la pregunta ni se integran las fuentes**: cada API genera su propio dataset independiente.

1. **Catálogo de juegos gratuitos:** qué títulos free-to-play y MMO existen, de qué género son y quién los publica.
2. **Oferta free-to-play por plataforma:** cómo se distribuye esa oferta gratuita, que es el modelo de monetización dominante en mobile.
3. **Contenido interno de un juego mobile:** el pool de cartas de *Yu-Gi-Oh! Duel Links*, como caso concreto del contenido que un título mobile pone a disposición del jugador.

## Consideraciones generales

- Debe utilizar **3 APIs diferentes** disponibles en: https://github.com/public-apis/public-apis
- Cada API debe aportar información relacionada con el mismo objetivo.
- Debe obtener **mínimo 200 registros por API**, salvo que la fuente disponga de menos registros en total.
- Cada API debe generar un archivo independiente en formato `.json`, `.xlsx`, `.csv` o `.txt`.
- **No realizar merge, join, concat ni cruces entre datasets.**
- El notebook debe poder ejecutarse nuevamente usando **Entorno de ejecución → Ejecutar todas**.

In [1]:
# Librerías base
import requests
import json
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path('.')

print(f'Carpeta de salida: {OUTPUT_DIR.resolve()}')

Carpeta de salida: /content


# Fuente 1 — API 1

**Nombre de la API:** MMOBomb Games API  
**Documentación:** https://www.mmobomb.com/api  
**Endpoint utilizado:** `https://www.mmobomb.com/api1/games`  
**Descripción de los datos:** catálogo de videojuegos free-to-play y MMO (aprox. 600 registros). Cada registro incluye título, género, plataforma, publisher, developer, fecha de lanzamiento y descripción corta.  
**Relación con el objetivo:** entrega la vista de **catálogo general de juegos gratuitos**, permitiendo caracterizar qué géneros y plataformas concentran la oferta free-to-play que también domina en mobile.

La API es pública, no requiere API key y no usa paginación: devuelve la lista completa en una sola llamada.

In [2]:
# CONFIGURACIÓN API 1
API1_URL = 'https://www.mmobomb.com/api1/games'
API1_MIN_REGISTROS = 200

api1_headers = {'User-Agent': 'MLY1101-Taller1'}
api1_params = {}

In [3]:
# CONSUMO API 1
api1_registros = []

response = requests.get(API1_URL, headers=api1_headers, params=api1_params, timeout=30)
print('Status API 1:', response.status_code)
response.raise_for_status()

api1_registros = response.json()      # la respuesta ya es la lista de juegos

print('Registros API 1:', len(api1_registros))

Status API 1: 200
Registros API 1: 414


In [4]:
# GUARDAR DATASET API 1
API1_ARCHIVO = OUTPUT_DIR / 'dataset_api_1.csv'

df_api1 = pd.DataFrame(api1_registros)
df_api1.to_csv(API1_ARCHIVO, index=False, encoding='utf-8-sig')

print('Archivo generado:', API1_ARCHIVO)
print('Registros guardados:', len(df_api1))
print('Dimensiones (filas, columnas):', df_api1.shape)

Archivo generado: dataset_api_1.csv
Registros guardados: 414
Dimensiones (filas, columnas): (414, 11)


# Fuente 2 — API 2

**Nombre de la API:** FreeToGame  
**Documentación:** https://www.freetogame.com/api-doc  
**Endpoint utilizado:** `https://www.freetogame.com/api/games`  
**Descripción de los datos:** listado completo de videojuegos free-to-play (aprox. 400–600 registros). Cada registro incluye título, género, plataforma, publisher, developer, fecha de lanzamiento y descripción corta.  
**Relación con el objetivo:** entrega la vista del **modelo free-to-play**, que es el esquema de monetización dominante en mobile. Permite caracterizar qué géneros concentran la oferta gratuita.

La API no usa paginación: devuelve la lista completa en una sola llamada.

In [5]:
# CONFIGURACIÓN API 2
API2_URL = 'https://www.freetogame.com/api/games'
API2_MIN_REGISTROS = 200

api2_headers = {'User-Agent': 'MLY1101-Taller1'}
api2_params = {}

In [6]:
# CONSUMO API 2
api2_registros = []

response = requests.get(API2_URL, headers=api2_headers, params=api2_params, timeout=30)
print('Status API 2:', response.status_code)
response.raise_for_status()

data = response.json()
api2_registros = data          # la respuesta ya es la lista de juegos

print('Registros API 2:', len(api2_registros))

Status API 2: 200
Registros API 2: 415


In [7]:
# GUARDAR DATASET API 2
API2_ARCHIVO = OUTPUT_DIR / 'dataset_api_2.csv'

df_api2 = pd.DataFrame(api2_registros)
df_api2.to_csv(API2_ARCHIVO, index=False, encoding='utf-8-sig')

print('Archivo generado:', API2_ARCHIVO)
print('Registros guardados:', len(df_api2))
print('Dimensiones (filas, columnas):', df_api2.shape)

Archivo generado: dataset_api_2.csv
Registros guardados: 415
Dimensiones (filas, columnas): (415, 11)


# Fuente 3 — API 3

**Nombre de la API:** YGOPRODeck (Yu-Gi-Oh! API)  
**Documentación:** https://ygoprodeck.com/api-guide/  
**Endpoint utilizado:** `https://db.ygoprodeck.com/api/v7/cardinfo.php?format=Duel%20Links`  
**Descripción de los datos:** cartas disponibles en el formato *Duel Links*, el juego mobile de Konami. Cada registro incluye nombre, tipo, atributo, nivel, ataque, defensa, arquetipo y descripción de efecto.  
**Relación con el objetivo:** entrega la vista de **contenido interno de un juego mobile**. El filtro `format=Duel Links` restringe el resultado al pool jugable del título móvil y no a Yu-Gi-Oh! en general.

In [8]:
# CONFIGURACIÓN API 3
API3_URL = 'https://db.ygoprodeck.com/api/v7/cardinfo.php'
API3_MIN_REGISTROS = 200

api3_headers = {}
api3_params = {'format': 'Duel Links'}

In [9]:
# CONSUMO API 3
api3_registros = []

response = requests.get(API3_URL, headers=api3_headers, params=api3_params, timeout=30)
print('Status API 3:', response.status_code)
response.raise_for_status()

data = response.json()
api3_registros = data['data']      # la lista de cartas viene bajo la clave 'data'

print('Registros API 3:', len(api3_registros))

Status API 3: 200
Registros API 3: 8230


In [10]:
# GUARDAR DATASET API 3
# json_normalize aplana los campos anidados de cada carta (card_images, card_prices, card_sets).
API3_ARCHIVO = OUTPUT_DIR / 'dataset_api_3.csv'

df_api3 = pd.json_normalize(api3_registros)
df_api3.to_csv(API3_ARCHIVO, index=False, encoding='utf-8-sig')

print('Archivo generado:', API3_ARCHIVO)
print('Registros guardados:', len(df_api3))
print('Dimensiones (filas, columnas):', df_api3.shape)

Archivo generado: dataset_api_3.csv
Registros guardados: 8230
Dimensiones (filas, columnas): (8230, 25)


# Resumen final

La siguiente celda debe ejecutarse al final y mostrar el resultado real de la carga.

In [11]:
total = len(api1_registros) + len(api2_registros) + len(api3_registros)

print('RESUMEN DE CARGA')
print('-' * 50)
print(f'API 1 (MMOBomb):    {len(api1_registros)} registros - {API1_ARCHIVO.name}')
print(f'API 2 (FreeToGame): {len(api2_registros)} registros - {API2_ARCHIVO.name}')
print(f'API 3 (YGOPRODeck): {len(api3_registros)} registros - {API3_ARCHIVO.name}')
print('-' * 50)
print(f'TOTAL: {total} registros')

if len(api1_registros) < 200:
    print('ADVERTENCIA: API 1 tiene menos de 200 registros. Documente la excepción si corresponde.')
if len(api2_registros) < 200:
    print('ADVERTENCIA: API 2 tiene menos de 200 registros. Documente la excepción si corresponde.')
if len(api3_registros) < 200:
    print('ADVERTENCIA: API 3 tiene menos de 200 registros. Documente la excepción si corresponde.')

RESUMEN DE CARGA
--------------------------------------------------
API 1 (MMOBomb):    414 registros - dataset_api_1.csv
API 2 (FreeToGame): 415 registros - dataset_api_2.csv
API 3 (YGOPRODeck): 8230 registros - dataset_api_3.csv
--------------------------------------------------
TOTAL: 9059 registros
